In [10]:
import re
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import spearmanr

import torch
import sys

device = "cuda" if torch.cuda.is_available() else "cpu"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from scripts.dataset_class import SimpleDataset
from scripts.preprocessing import Preprocessor

import nested_cv_metrics_opt as n_cv_opt

# sys.path.insert(0, "/beegfs/home/f.gubanov/f.gubanov/bimai_lab/metrics_code_v2/scripts")
from scripts.normalization_diagnostics import diagnose_normalization

In [12]:
oof_preds, fold_meta, meta_df, info = n_cv_opt.load_artifact("results/check_fm.pkl")
info

Loaded artifact from results/check_fm.pkl
  run_info: {'timestamp': '2026-09-23 00:45:28', 'outer_seed': 155, 'identities': ['phikon_spec', 'hibou_l_spec'], 'configs': "normalization:[False, True] | channel_mode:['gray', 'hed'] | flip_intensity:[False, True] | match_histogram:[False, True] | clahe:[False, True] | smoothing:[False, True]", 'n_outer_splits': 5, 'n_inner_splits': 4}


{'timestamp': '2026-09-23 00:45:28',
 'outer_seed': 155,
 'identities': ['phikon_spec', 'hibou_l_spec'],
 'configs': "normalization:[False, True] | channel_mode:['gray', 'hed'] | flip_intensity:[False, True] | match_histogram:[False, True] | clahe:[False, True] | smoothing:[False, True]",
 'n_outer_splits': 5,
 'n_inner_splits': 4}

In [13]:
fm_df = pd.DataFrame(fold_meta).sort_values(by='metric')
fm_df = n_cv_opt.add_preproc_key(fm_df)
fm_df.groupby("metric")["preproc_cfg"].value_counts()

metric        preproc_cfg                             
hibou_l_spec  0 | rgb | 0 | 0 | 0 | 0 | cos | layer_20    3
              0 | rgb | 0 | 0 | 0 | 0 | cos | layer_21    2
phikon_spec   0 | rgb | 0 | 1 | 0 | 0 | cos | layer_1     4
              1 | rgb | 0 | 1 | 0 | 0 | cos | layer_1     1
Name: preproc_cfg, dtype: int64

In [14]:
pd.set_option('display.max_colwidth', None)
diag = diagnose_normalization(oof_preds, fold_meta, threshold=1.5)
diag

,metric,fold_mean_max,fold_mean_min,mean_max,mean_min,mean_abs_diff,std_max,std_min,mean_diff/std,normalize,cfg_diff_desc
0,hibou_l_spec,3,0,0.3943,0.3164,0.0779,0.1022,0.0542,0.7622,False,feature: layer_21→layer_20
1,phikon_spec,4,1,0.6706,0.6050,0.0656,0.0705,0.0393,0.9305,False,normalization: False→True


In [15]:
# rank-нормализация (решает проблему разброса между фолдами)
# oof_preds = n_cv_opt.normalize_oof_per_fold(oof_preds, metrics = None)
# --- результаты ---
results = n_cv_opt.evaluate_oof(oof_preds, meta_df)
bs = n_cv_opt.bootstrap_oof(oof_preds, meta_df)
results

,metric,spearman_class3,auc_bad_vs_rest,auc_good_vs_rest,auc_3class_ovo,spearman_raw_1_5
0,hibou_l_spec,0.665967,0.883782,0.824223,0.812033,0.682881
1,phikon_spec,0.600341,0.836147,0.803162,0.779425,0.598930


**Proposed HAPS config x metric:** 

- Backbone = `hibou_l`
- Config: {*normalization: False, channel_mode: `rgb`, flip_intensity: False, match_histogram: False, clahe: False, smoothing: False, agg: `cos`, feature: `layer_20`*}

In [61]:
%%bash
source slurm/env.sh
.venv-foundation/bin/python -c '
## Фиксированный конфиг
fixed_cfg = {
    "normalization": False,
    "channel_mode": "rgb",
    "flip_intensity": False,
    "match_histogram": False,
    "clahe": False,
    "smoothing": False,
    "agg": "cos",
    "feature": "layer_20",
    "metric": "hibou_l_spec",
}

# План скрипта (plain 5-fold, без inner CV)
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from scripts.dataset_class import SimpleDataset
import nested_cv_metrics_opt as ncv

csv = "/beegfs/home/f.gubanov/f.gubanov/bimai_lab/filtration_imgs/exp_full.csv"
ds = SimpleDataset(csv, dist_col="Similarity_Score", class_col="class3", return_meta=True)
all_pairs, meta_df = ncv._cache_dataset(ds)

y = meta_df["class3"].to_numpy(int)
groups = meta_df["pname"].astype(str).to_numpy()
splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=155)
              .split(np.zeros(len(meta_df)), y, groups))

oof = []
for fold_id, (tr, va) in enumerate(splits):
    val_pairs = [all_pairs[i] for i in va]
    scores = ncv.compute_scores_for_config(fixed_cfg, val_pairs, "hibou_l_spec")  # np.ndarray
    for j, idx in enumerate(va):
        oof.append({"sample_idx": int(idx), "metric": "hibou_l_spec",
                    "fold": int(fold_id), "similarity_score": float(scores[j]),
                    "config": fixed_cfg})

print(ncv.evaluate_oof(oof, meta_df).to_string())       # точечные метрики
print(ncv.bootstrap_oof(oof, meta_df).to_string())      # CI95
'

REPO=/beegfs/home/f.gubanov/f.gubanov/bimai_lab/HAPS_v1_Vladislav
PYTHON=/beegfs/home/f.gubanov/f.gubanov/bimai_lab/HAPS_v1_Vladislav/.venv/bin/python
torch 2.2.2+cu121 cuda True


/beegfs/home/f.gubanov/f.gubanov/bimai_lab/HAPS_v1_Vladislav/.venv-foundation/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Caching: 100%|██████████| 512/512 [00:03<00:00, 146.11it/s]


Cached 512 pairs, 30 WSI
[foundation:hibou_l] input spec: size=224 crop_pct=1.0 interp=bicubic mean=(0.7068, 0.5755, 0.722) std=(0.195, 0.2316, 0.1816)
         metric  spearman_class3  auc_bad_vs_rest  auc_good_vs_rest  auc_3class_ovo  spearman_raw_1_5
0  hibou_l_spec         0.672799          0.88925          0.825851        0.814938          0.690061
         metric  full_spearman_class3  full_auc_bad_vs_rest  full_auc_good_vs_rest  full_auc_3class_ovo  full_spearman_raw_1_5  bs_sp_mean  bs_sp_std        bs_sp_ci  bs_auc_bad_mean  bs_auc_bad_std  bs_auc_bad_ci  bs_auc_good_mean  bs_auc_good_ci  bs_auc_ovo_mean   bs_auc_ovo_ci  n_boot  seed
0  hibou_l_spec              0.672799               0.88925               0.825851             0.814938               0.690061    0.660831   0.059824  (0.544, 0.768)         0.886429        0.027343  (0.83, 0.935)          0.819866  (0.737, 0.901)         0.810434  (0.747, 0.873)    1000   143


### Concatenate multiple FM (`hoptipus0`, `gigapath`, `uni2h`) results

In [7]:
fm_backbones = ["hoptimus0_spec", "uni2h_spec", "gigapath_spec"]
oof_all, fold_all = [], []
for fm_b in fm_backbones:
    oof, fold, _, _ = n_cv_opt.load_artifact(f"results/check_fm_{fm_b}.pkl")
    oof_all.extend(oof)
    fold_all.extend(fold)

diag = diagnose_normalization(oof_all, fold_all, threshold=1.5)
print("=== Normalization diagnostics (ratio = mean_abs_diff / std_max) ===")
diag

Loaded artifact from results/check_fm_hoptimus0_spec.pkl
  run_info: {'timestamp': '2026-09-23 18:25:07', 'outer_seed': 155, 'identities': ['hoptimus0_spec'], 'configs': "normalization:[False, True] | channel_mode:['gray', 'hed'] | flip_intensity:[False, True] | match_histogram:[False, True] | clahe:[False, True] | smoothing:[False, True]", 'n_outer_splits': 5, 'n_inner_splits': 4}
Loaded artifact from results/check_fm_uni2h_spec.pkl
  run_info: {'timestamp': '2026-09-23 21:57:44', 'outer_seed': 155, 'identities': ['uni2h_spec'], 'configs': "normalization:[False, True] | channel_mode:['gray', 'hed'] | flip_intensity:[False, True] | match_histogram:[False, True] | clahe:[False, True] | smoothing:[False, True]", 'n_outer_splits': 5, 'n_inner_splits': 4}
Loaded artifact from results/check_fm_gigapath_spec.pkl
  run_info: {'timestamp': '2026-09-25 02:56:30', 'outer_seed': 155, 'identities': ['gigapath_spec'], 'configs': "normalization:[False, True] | channel_mode:['gray', 'hed'] | flip_int

,metric,fold_mean_max,fold_mean_min,mean_max,mean_min,mean_abs_diff,std_max,std_min,mean_diff/std,normalize,cfg_diff_desc
0,hoptimus0_spec,3,2,0.6782,-0.9821,1.6603,0.1150,0.0425,14.4374,True,agg: cos→dist; feature: layer_14→mean; normali...
1,uni2h_spec,4,0,0.7175,-0.8231,1.5406,0.0638,0.0295,24.1473,True,agg: cos→dist; feature: layer_19→mean; match_h...
2,gigapath_spec,3,0,0.9848,0.4657,0.5191,0.0834,0.0076,6.2242,True,feature: layer_0→layer_39; match_histogram: Tr...


In [9]:
# rank-нормализация (решает проблему разброса между фолдами)
oof_all = n_cv_opt.normalize_oof_per_fold(oof_all, metrics = None)
# --- результаты ---
res_all = n_cv_opt.evaluate_oof(oof_all, meta_df)
bs_all = n_cv_opt.bootstrap_oof(oof_all, meta_df)
res_all

,metric,spearman_class3,auc_bad_vs_rest,auc_good_vs_rest,auc_3class_ovo,spearman_raw_1_5
0,uni2h_spec,0.549310,0.820165,0.763425,0.753529,0.552163
1,gigapath_spec,0.470113,0.771757,0.727939,0.712548,0.475638
2,hoptimus0_spec,0.428080,0.747182,0.707866,0.690995,0.422376


### Recalculating Vladislav's results

In [55]:
oof_preds, fold_meta, meta_df, info = n_cv_opt.load_artifact("results_vlad/cellpose.pkl")

Loaded artifact from results_vlad/cellpose.pkl
  run_info: {'timestamp': '2026-08-05 18:23:20', 'outer_seed': 155, 'identities': ['cellpose'], 'configs': "normalization:[False, True] | channel_mode:['gray', 'hed'] | flip_intensity:[False, True] | match_histogram:[False, True] | clahe:[False, True] | smoothing:[False, True] | binary_mask:[False]", 'n_outer_splits': 5, 'n_inner_splits': 4}


In [56]:
fm_df = pd.DataFrame(fold_meta).sort_values(by='metric')
fm_df = n_cv_opt.add_preproc_key(fm_df)
fm_df.groupby("metric")["preproc_cfg"].value_counts()

metric    preproc_cfg                                 
cellpose  1 | gray | 1 | 1 | 1 | 1 | 0 | layer_3 | cos    3
          0 | rgb | 1 | 1 | 1 | 0 | 0 | mean | cos        1
          1 | rgb | 1 | 1 | 1 | 1 | 0 | layer_3 | cos     1
Name: preproc_cfg, dtype: int64

In [45]:
pd.set_option('display.max_colwidth', None)
diag = diagnose_normalization(oof_preds, fold_meta, threshold=1.5)
diag

,metric,fold_mean_max,fold_mean_min,mean_max,mean_min,mean_abs_diff,std_max,std_min,mean_diff/std,normalize,cfg_diff_desc
0,cellpose,2,3,0.4958,0.3544,0.1414,0.1005,0.0295,1.407,False,channel_mode: rgb→gray; feature: mean→layer_3; normalization: False→True; smoothing: False→True


In [57]:
# rank-нормализация (решает проблему разброса между фолдами)
# oof_preds = n_cv_opt.normalize_oof_per_fold(oof_preds)
# --- результаты ---
res = n_cv_opt.evaluate_oof(oof_preds, meta_df)
# bs = n_cv_opt.bootstrap_oof(oof_preds, meta_df)
res

,metric,spearman_class3,auc_bad_vs_rest,auc_good_vs_rest,auc_3class_ovo,spearman_raw_1_5
0,cellpose,0.493938,0.778481,0.747732,0.731468,0.499632


```
sbatch --export=ALL,\
VENV_PY=/beegfs/home/f.gubanov/f.gubanov/bimai_lab/HAPS_v1_Vladislav/.venv-foundation/bin/python,\
HF_HUB_CACHE=/trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub,\
HUGGINGFACE_HUB_CACHE=/trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub,\
TRANSFORMERS_CACHE=/trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub,\
HF_HUB_OFFLINE=1 \
run_cv.sbatch "phikon_spec,hibou_l_spec" \
/beegfs/home/f.gubanov/f.gubanov/bimai_lab/HAPS_v1_Vladislav/results/check_fm.pkl
```

```
sbatch --export=ALL,VENV_PY=/beegfs/home/f.gubanov/f.gubanov/bimai_lab/HAPS_v1_Vladislav/.venv-foundation/bin/python \
run_cv.sbatch \
"hoptimus0_spec,uni2h_spec,gigapath_spec" \
/beegfs/home/f.gubanov/f.gubanov/bimai_lab/HAPS_v1_Vladislav/results/check_fm_hoptimus0_spec.pkl
```

`transpath` VS `ctranspath_hf`

| | transpath (deep_metrics) | ctranspath_hf (foundation_metrics)  |
|------|------|------|
| Модель | CTransPath (Swin-Tiny + ConvStem) | та же CTransPath |
| Веса	 | локальный TransPath/ctranspath.pth |	HF-зеркало 1aurent/swin_tiny_patch4_window7_224.CTransPath |
| Загрузка |	patched timm 0.5.4 + ctran.py |	modern timm>=1.0 + свой _CTransPathConvStem |
| Окружение |	.venv	| .venv-foundation |
| Вход	| _transpath_input: resize 224 + ImageNet norm	| prepare_input: resize+crop (crop_pct=0.9) + ImageNet norm|

В foundation_metrics.py прямо написано: ctranspath_hf — это «HF-зеркало; локальный TransPath остаётся в deep_metrics.transpath». Т.е. это те же веса, переупакованные под HF/timm.

Единственное содержательное отличие — препроцессинг: у ctranspath_hf crop_pct=0.9 (resize→249→center-crop 224, bicubic), у transpath — прямой resize в 224 (bilinear). Это может дать мелкий сдвиг, но не качественную разницу.

**Вывод: считать не стоит**

Если CTransPath нужен в финальной таблице на 512 для полноты — достаточно одного прогона:
- либо перегнать transpath на 512 (нужен .venv + patched timm + локальный ctranspath.pth);
- либо прогнать ctranspath_hf на 512 (проще: .venv-foundation + HF-кэш уже настроены, никаких patched-зависимостей).

Если нужна полнота — взять ctranspath_hf (удобнее).

### Тест загрузки FM модели из HF

```
$ ls -d /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--* 2>/dev/null
    /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--1aurent--swin_tiny_patch4_window7_224.CTransPath
    /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--bioptimus--H-optimus-0
    /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--histai--hibou-L
    /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--MahmoodLab--TITAN
    /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--MahmoodLab--UNI2-h
    /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--owkin--phikon
    /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--paige-ai--Virchow2
    /trinity/home/vladislav.kozlovskiy/.cache/huggingface/hub/models--prov-gigapath--prov-gigapath
```

В кэше коллеги лежат все foundation-модели: `CTransPath (ctranspath_hf)`, `H-optimus-0`, `hibou-L`, `TITAN`, `UNI2-h`, `phikon`, `Virchow2`, `gigapath`
Весь список FOUNDATION_BACKBONES (кроме cpsam, чьи веса в ~/.cellpose/models) уже скачан и грузится offline через трюк HF_HOME + HF_TOKEN_PATH. 

Для hibou_l_spec, phikon_spec (и вообще любых FM из списка) **токен не требуется**.

In [10]:
%%bash
export HF_HOME=/trinity/home/vladislav.kozlovskiy/.cache/huggingface
export HF_TOKEN_PATH=/trinity/home/f.gubanov/.cache/huggingface/token   # не существует -> no token
export HF_HUB_OFFLINE=1
export TRANSFORMERS_OFFLINE=1
.venv-foundation/bin/python scripts/verify_foundation.py --models hibou_l --device cpu --skip-random


hibou_l  (histai/hibou-L)
  declared mean=(0.7068, 0.5755, 0.722) std=(0.195, 0.2316, 0.1816) size=224
  ours     mean=(0.7068, 0.5755, 0.722) std=(0.195, 0.2316, 0.1816) size=224
  NORM OK | SIZE OK
[foundation:hibou_l] input spec: size=224 crop_pct=1.0 interp=bicubic mean=(0.7068, 0.5755, 0.722) std=(0.195, 0.2316, 0.1816)
  loaded: Dinov2ModelWithRegisters, params=303.7M
  embed dim=(1, 1024) (config num_features=1024)  norm=33.166
  self-cos=1.0000  unrelated-cos=0.9824


**Когда токен понадобится:**

1. Сейчас всё держится на /trinity/.../vladislav.kozlovskiy/.cache/.... Если /home коллеги почистят — кэш исчезнет. Свой токен + свой кэш снимают эту зависимость.
2. Новые gated-модели, которых нет в кэше коллеги.
3. cpsam (Cellpose-SAM) — если захотите его гонять.

1. Сохраняем токен huggingface

In [9]:
# %%bash
# .venv-foundation/bin/huggingface-cli login --token hf_xxxxxxxx

2. Проверить работоспособность БЕЗ загрузки весов ( доступ к gated hibou-L (метаданные, без весов))

In [8]:
%%bash
.venv-foundation/bin/huggingface-cli whoami

.venv-foundation/bin/python -c "
from huggingface_hub import HfApi
from huggingface_hub.errors import GatedRepoError
try:
    info = HfApi().model_info('histai/hibou-L')
    print('OK: доступ есть,', info.id)
except GatedRepoError as e:
    print('GATED: лицензия не принята ->', e)
except Exception as e:
    print('ERROR:', e)
"

Teooo
OK: доступ есть, histai/hibou-L
